# 将自然语言描述转化为数据库的SQL查询语句

## 1. 数据集
b-mc2/sql-create-context  

### 指令数据预处理
原始数据如下：
```
question为How many heads of the departments are older than 56 ?, context为CREATE TABLE head (age INTEGER), answer为SELECT COUNT(*) FROM head WHERE age > 56。
```
预处理为指令数据如下：  
question为用户问题，context为数据库表格创建语句，answer为最终产生的SQL语句
```
\nBelow is an instruction that describes a task.Write a response that appropriately completes the request.\n### Instruction: How many heads of the departments are older than 56 ?\n### Database Schema:\nCREATE TABLE head (age INTEGER)\n### Response:\nSELECT COUNT(*) FROM head WHERE age > 56\n<eos>\n

```

## 2. 模型微调

In [ ]:
from datasets import load_dataset
import torch
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Hugging Face model id
model_id = "./models/gemma-7b"

# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'right'

train_dataset = load_dataset(
    "json", data_files="sql-create-context.json")['train']
print(train_dataset[0])
print(f"train size: {len(train_dataset)}")

# LoRA config based on QLoRA paper & Sebastian Raschka experiment
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=64,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
    task_type="CAUSAL_LM",
)

args = TrainingArguments(
    output_dir="output",                    # directory to save and repository id
    num_train_epochs=2,                     # number of training epochs
    per_device_train_batch_size=8,          # batch size per device during training
    # number of steps before performing a backward/update pass
    gradient_accumulation_steps=4,
    # use gradient checkpointing to save memory
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,                       # log every 10 steps
    bf16=True,                              # use bfloat16 precision
    learning_rate=1e-4,                     # learning rate, based on QLoRA paper
    max_grad_norm=0.3,                      # max gradient norm based on QLoRA paper
    warmup_ratio=0.1,                      # warmup ratio based on QLoRA paper
    lr_scheduler_type="constant",           # use constant learning rate scheduler
    push_to_hub=False,                       # push model to hub
    report_to="tensorboard",                # report metrics to tensorboard
)

max_seq_length = 1024

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    packing=False,
    dataset_text_field="text"
)


trainer.train()
trainer.save_model()

## 3. 模型测试

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

peft_model_id = "./output/checkpoint-4911"
model = AutoModelForCausalLM.from_pretrained(peft_model_id, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained("./models/gemma-7b")


while True:
    question = input("enter a question: ")
    context = input("enter database schema: ")
    input_text = f"""
Below is an instruction that describes a task.Write a response that appropriately completes the request.
### Instruction: {question}
### Database Schema:
{context}
### Response:
"""
    encoding = tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **encoding, max_new_tokens=100, temperature=0.1, do_sample=True)
    generated_ids = outputs[:, encoding.input_ids.shape[1]:]
    generated_texts = tokenizer.batch_decode(
        generated_ids, skip_special_tokens=True)
    print("Instruction: ", input_text)
    print("SQL: ", generated_texts[0].strip())

question: How many heads of the departments are older than 56